<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">شش گام پیوسته یا چهار گام و ادامه؟</h1>
<p style="text-align:right">درس 58 از 92 · چطور ادامهٔ اجرا را با اجرای پیوسته مقایسه کنیم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">51-resume</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-02/51-resume.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">دو اجرای واقعی را مقایسه کنید و وضعیت تصادفی انتخاب <bdi dir="ltr">Batch</bdi> را بازیابی کنید.</p><p style="text-align:right">پیش‌نیاز: <bdi dir="ltr">Checkpoint</bdi>، <bdi dir="ltr">AdamW</bdi> و مولد عدد تصادفی.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۴۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر وزن‌ها برابر باشند ولی وضعیت مولد <bdi dir="ltr">Batch</bdi> فرق کند، آیا ادامهٔ آموزش الزاماً برابر است؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import tempfile
from pathlib import Path
import torch
from mini_gpt.train import build_parser,train
from mini_gpt.checkpoint import load_checkpoint
torch.set_num_threads(1)
parser = build_parser()
with tempfile.TemporaryDirectory(prefix='aibook-resume-') as directory:
    root = Path(directory)
    corpus = root/'corpus.txt'
    corpus.write_text('abcde '*20,encoding='utf-8')
    common = ['--text',str(corpus),'--context-length','4','--embedding-dim','8',
              '--num-heads','2','--num-layers','1','--batch-size','2','--dropout','0.1',
              '--threads','1','--device','cpu','--eval-every','2','--seed','17']
    full_path = train(parser.parse_args(common+['--steps','6','--output',str(root/'full')]))
    split_path = train(parser.parse_args(common+['--steps','4','--output',str(root/'split')]))
    resumed_path = train(parser.parse_args(['--text',str(corpus),'--resume',str(split_path),
        '--output',str(root/'split'),'--steps','6','--eval-every','2','--threads','1','--device','cpu']))
    _,_,full_payload = load_checkpoint(full_path)
    _,_,resumed_payload = load_checkpoint(resumed_path)
print('total steps:',full_payload['step'],resumed_payload['step'])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">different_tensors(left, right)</code> برای دو <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">state_dict</code> با کلیدهای برابر، نام <bdi dir="ltr">Tensor</bdi>های نابرابر را به‌ترتیب الفبایی برگرداند. از برابری کامل استفاده کنید، نه <bdi dir="ltr">Loss</bdi> گرد‌شده.</p>
</div>

In [ ]:
def different_tensors(left, right):
    # TODO: فهرست نام Tensorهای متفاوت
    return None

In [ ]:
def test_exercise():
    result = different_tensors(full_payload['model'],resumed_payload['model'])
    if result is None:
        return False
    assert result==[],result
    assert different_tensors({'b':torch.tensor([1]),'a':torch.tensor([2])},
                             {'b':torch.tensor([2]),'a':torch.tensor([3])})==['a','b']
    assert different_tensors({'x':torch.tensor([1.0])},{'x':torch.tensor([1.0001])})==['x']
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: different_tensors')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط وضعیت مولد انتخاب <bdi dir="ltr">Batch</bdi> را عوض کنید؛ محدوده و تعداد نمونه‌ها ثابت بمانند.</p>
</div>

In [ ]:
for seed in (17,18):
    generator = torch.Generator().manual_seed(seed)
    print(seed,torch.randint(100,(8,),generator=generator).tolist())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">ساخت دوبارهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Generator</code> با <bdi dir="ltr">Seed</bdi> اولیه، وضعیت میانهٔ اجرا را برنمی‌گرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">restore_batch_generator(payload)</code> یک <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Generator</code> تازه با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">rng_batches</code> ذخیره‌شده بسازد.</p>
</div>

In [ ]:
wrong = torch.Generator().manual_seed(18)
saved = torch.Generator()
saved.set_state(resumed_payload['rng_batches'])
print('from initial seed:',torch.randint(100,(8,),generator=wrong).tolist())
print('from saved state:',torch.randint(100,(8,),generator=saved).tolist())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def restore_batch_generator(payload):
    # TODO: از وضعیت ذخیره‌شده، نه Seed آغازین
    return None

In [ ]:
def test_repair():
    generator = restore_batch_generator(resumed_payload)
    if generator is None:
        return False
    expected = torch.Generator()
    expected.set_state(resumed_payload['rng_batches'])
    assert torch.equal(torch.randint(100,(12,),generator=generator),torch.randint(100,(12,),generator=expected))
    another = restore_batch_generator(resumed_payload)
    assert torch.equal(another.get_state(),resumed_payload['rng_batches'])
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: restore_batch_generator')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">هر سه اجرا از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">train</code> واقعی‌اند؛ فایل‌ها در پوشه‌های موقت مستقل ساخته و سپس پاک شدند. برابری بیت‌به‌بیت این آزمون مربوط به <bdi dir="ltr">CPU</bdi> و محیط ثابت همین اجراست، نه وعده‌ای برای هر <bdi dir="ltr">GPU</bdi> یا نسخه.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">علاوه بر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">rng_batches</code>، کدام وضعیت‌های <bdi dir="ltr">Checkpoint</bdi> برای ادامهٔ دقیق لازم‌اند؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-02/51-resume.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/51-resume.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>